In [1]:
!pip install streamlit pyngrok joblib



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 20.1 MB/s eta 0:00:00
  Attempting uninstall: cachetools
    Found existing installation: cachetools 7.0.0
    Uninstalling cachetools-7.0.0:
      Successfully uninstalled cachetools-7.0.0


In [2]:
!ngrok authtoken 36sYJhePM8fSgorbGD9qwezBXtH_6Grfm2acreBuNtP3VYzYv


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [3]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib

# Load trained Ridge Regression model, scaler, and feature names
model = joblib.load('final_ridge_model.pkl')
scaler = joblib.load('scaler.pkl')
feature_names = joblib.load('feature_names.pkl')

# Page setup
st.set_page_config(
    page_title="Agri-food CO₂ Emission Predictor",
    layout="wide"
)

st.title("🌍 Agri-food CO₂ Emission Prediction App")
st.write(
    "This application predicts total agrifood CO₂ emissions "
    "using environmental, agricultural, and population indicators."
)

# Sidebar inputs
st.sidebar.header("Input Features")
user_input = {}
for feature in feature_names:
    user_input[feature] = st.sidebar.number_input(
        feature,
        value=0.0,
        format="%.4f"
    )

# Convert input to DataFrame and scale
input_df = pd.DataFrame([user_input])
input_scaled = scaler.transform(input_df)

# Predict button
if st.button("Predict CO₂ Emissions"):
    prediction = model.predict(input_scaled)[0]
    st.success(f"🌱 Predicted Total CO₂ Emission: **{prediction:,.2f} CO₂eq**")

    # Feature importance chart
    coef = pd.Series(model.coef_, index=feature_names)
    st.subheader("🔍 Feature Importance")
    st.bar_chart(coef.sort_values(key=abs))


Writing app.py


In [6]:
# Kill old Streamlit processes
!kill $(ps aux | grep streamlit | awk '{print $2}') 2>/dev/null

# Run Streamlit headless
import time
import os

os.system("nohup streamlit run app.py --server.port 8501 --server.headless true &")
time.sleep(15)  # give Streamlit a few seconds to start

# Connect ngrok
from pyngrok import ngrok
public_url = ngrok.connect(8501)
public_url



^C


<NgrokTunnel: "https://anastacia-overprominent-retta.ngrok-free.dev" -> "http://localhost:8501">